
# 6.2 🔌 Endpoints, Parámetros y Pydantic en FastAPI

En el notebook anterior aprendimos:

- qué es una API
- cómo crear un servidor FastAPI
- cómo ejecutar la aplicación con `uvicorn`

Ahora vamos a aprender a construir **APIs útiles**, capaces de recibir datos, validarlos y devolver resultados estructurados.

---

## 🎯 Objetivos

En este notebook aprenderás:

- qué es un **endpoint**
- cómo usar **path parameters**
- cómo usar **query parameters**
- cómo recibir datos en el **body**
- cómo validar datos usando **Pydantic**
- cómo definir **response models**



# 1️⃣ ¿Qué es un endpoint?

Un **endpoint** es una ruta de la API que ejecuta una función.

Por ejemplo:

```
GET /health
GET /instrument/AAPL
POST /portfolio/analyze
```

Cada endpoint representa **una operación específica del sistema**.



# 2️⃣ Endpoint con Path Parameter

Los **path parameters** permiten enviar información directamente en la URL.

Ejemplo:

```
GET /instrument/AAPL
```

Aquí `AAPL` es el parámetro.


In [1]:

from fastapi import FastAPI

app = FastAPI()

@app.get("/instrument/{ticker}")
def get_instrument(ticker: str):
    return {
        "ticker": ticker,
        "message": f"Información solicitada para {ticker}"
    }



Si visitas:

```
/instrument/TSLA
```

la respuesta será:

```json
{
 "ticker": "TSLA",
 "message": "Información solicitada para TSLA"
}
```



# 3️⃣ Query Parameters

Los **query parameters** se envían después del símbolo `?` en la URL.

Ejemplo:

```
GET /history/AAPL?days=30
```


In [2]:

@app.get("/history/{ticker}")
def get_history(ticker: str, days: int = 30):
    return {
        "ticker": ticker,
        "days_requested": days
    }



Aquí:

- `ticker` es un **path parameter**
- `days` es un **query parameter**



# 4️⃣ Request Body

Cuando enviamos información compleja usamos el **body de la petición**.

Por ejemplo:

```
POST /portfolio/analyze
```



Aquí es donde entra **Pydantic**.



# 5️⃣ Pydantic: Validación de datos

FastAPI usa **Pydantic** para validar datos automáticamente.

Pydantic nos permite definir **modelos estructurados**.


In [3]:

from pydantic import BaseModel
from typing import List

class PortfolioRequest(BaseModel):
    tickers: List[str]
    weights: List[float]



Este modelo define el formato esperado:

```json
{
 "tickers": ["AAPL","MSFT","NVDA"],
 "weights": [0.4,0.3,0.3]
}
```



# 6️⃣ Endpoint usando Pydantic


In [4]:

@app.post("/portfolio/analyze")
def analyze_portfolio(data: PortfolioRequest):
    
    total_weight = sum(data.weights)
    
    return {
        "tickers": data.tickers,
        "total_weight": total_weight,
        "valid_portfolio": abs(total_weight - 1.0) < 0.01
    }



FastAPI automáticamente:

- valida el JSON
- convierte tipos
- genera documentación



# 7️⃣ Response Models

También podemos definir **modelos de salida**.


In [5]:

class PortfolioResponse(BaseModel):
    tickers: List[str]
    total_weight: float
    valid_portfolio: bool


In [6]:

@app.post("/portfolio/validate", response_model=PortfolioResponse)
def validate_portfolio(data: PortfolioRequest):
    
    total_weight = sum(data.weights)
    
    return PortfolioResponse(
        tickers=data.tickers,
        total_weight=total_weight,
        valid_portfolio=abs(total_weight - 1.0) < 0.01
    )



Esto garantiza que la API siempre devuelva **estructuras consistentes**.



# 8️⃣ Flujo completo de una API

```mermaid
flowchart LR
    A[Cliente] --> B[Request JSON]
    B --> C[FastAPI Endpoint]
    C --> D[Pydantic Validation]
    D --> E[Lógica del Servicio]
    E --> F[Response Model]
    F --> A
```



# 9️⃣ Cómo integrar esto con SmartPortfolio

En los próximos notebooks usaremos endpoints como:

```
POST /portfolio/optimize
GET /portfolio/recommendations
POST /portfolio/backtest
```

Estos endpoints usarán:

- `Portfolio`
- `PortfolioOptimizer`
- `MarketDataProvider`



# 🧠 Ideas clave

Después de este notebook debes entender:

- cómo crear endpoints
- cómo usar path parameters
- cómo usar query parameters
- cómo validar datos con Pydantic
- cómo estructurar respuestas

---

En el siguiente notebook construiremos una **arquitectura real de API**, separando:

- routers
- schemas
- services
- configuración
